# Exercise 1: How Does an LLM API Call Work?

This notebook shows the basics of how we talk to a large language model and how "tool calling" actually works under the hood. By the end you should see that there is no magic: an LLM only generates text, and everything else (memory, tools, structured fields) is code that wraps around it.

What you will work through:

1. **A Single LLM Call**: the ollama wrapper, chat templates, and how `thinking` and `content` are parsed out of raw text.
2. **Statelessness**: the API has no memory; a "conversation" is just the history you resend on each call.
3. **Tool Calling**: tools are not a special model feature, just structured text output the model was trained to produce.
4. **The Tool-Use Loop**: the cycle an agent automates (call, check, execute, send result back, call again).

Prerequisite: complete the README setup. If any of this feels fast, work through Exercise 0 first.

Verify ollama is running and the model is loaded. If this fails, nothing below will work.

In [ ]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b not found: {models}"
print("OK")

## Part 1: A Single LLM Call

This part is about the **ollama wrapper** we use to call the model. To see what ollama actually does for you, and why a wrapper is needed at all, we will look inside `ollama.chat()` step by step.

This is the one place in the lab where we go *below* `ollama.chat()`. Exercise 0 kept you inside that clean call on purpose; Part 1 peels it open once so you can see there is no magic, and from Part 2 on we use `ollama.chat()` for good. (A **chat template** is the fixed text format, with special marker tokens, that a model was trained to read conversations in.)

An LLM only generates raw text. The convenient fields you see (`thinking`, `content`, `tool_calls`) are parsed out of that text after generation. Inside `ollama.chat()` three layers do this work:

1. **Render**: serializes your `messages` into a single prompt string using the model's chat template.
2. **Complete**: the model produces raw text, token by token, until it emits an end-of-turn token.
3. **Parse**: splits the raw output on `<think>...</think>` into `thinking` and `content`. Tool calls (Part 3) work the same way.

```
  messages list           raw text stream             ChatResponse
  ─────────────           ───────────────             ─────────────
  [{role:"user",          <|im_start|>user            {
    content: ...},         What is 2+2?                 thinking: "...",
   {role:"assistant",      <|im_end|>                   content: "4",
    content: ...}]         <|im_start|>assistant        tool_calls: None
                           <think>...</think>          }
                           4
                           <|im_end|>

      RENDER          →        COMPLETE          →        PARSE
  (apply template)         (model generates)         (regex split on
                                                       <think>...)
```

In the cells below we call the lower-level `/api/generate` endpoint, so you can see the raw text directly, before parsing.

<details><summary>Where this lives in the ollama source</summary>

`renderPrompt` → `r.Completion` → `Qwen35Parser.Add` inside [ollama's `ChatHandler`](https://github.com/ollama/ollama/blob/main/server/routes.go).

</details>

Raw output via `/api/generate`. The prompt below uses Qwen3.5's chat template:

- `<|im_start|>user\n...<|im_end|>` for a user turn
- `<|im_start|>assistant\n...<|im_end|>` for the model's reply
- `<|im_start|>system\n...<|im_end|>` for system instructions
- `<think>...</think>` for reasoning, separate from the answer

**`<|im_end|>` is the model's stop token.** When the model emits it during generation, the API ends the stream and returns the output.

Prefilling `<think>\n` makes the model start inside the thinking block.

**Goal:** see the unparsed text. `thinking` and `content` are produced by splitting this stream, not returned as separate fields.

<details><summary>What are these tokens, and why are they model-specific?</summary>

The strings above (`<|im_start|>`, `<|im_end|>`, `<think>`) are **special tokens**: short markers the model was trained to recognize as conversation structure. You normally never see them, since `ollama.chat()` and cloud APIs hide them behind a clean `{role, content}` interface.

They are **model-specific**. Qwen3.5 uses the ChatML format. Llama, Mistral, and other model families have their own conventions, baked in at training time. A template written for one model will not work for another, and using the wrong template silently degrades output quality.

</details>

<details><summary>Why prompt injection works</summary>

These tokens are just text the model was trained to recognize. If user-supplied content contains a fake `<|im_end|>` or `<|im_start|>system`, the model cannot distinguish it from the trusted template tokens. Production code usually sanitizes user input to prevent exactly this.

</details>

In [ ]:
# CELL: raw-API prefill
# Raw LLM output via the REST /api/generate endpoint: the lower-level call that
# ollama.chat() uses internally. raw=True skips the render + parse layers, so we
# see the unparsed text exactly as the model generates it.
import requests, json

# (plumbing) the local Ollama address, e.g. http://localhost:11434.
# You do not need to understand this line.
OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWhat is 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# We pre-filled "<think>\n" as the start of the assistant's reply. Because an LLM
# only continues text (Exercise 0), it carries on from inside the <think> block
# and never re-emits that opening tag, so we prepend it back here.
raw_output = "<think>\n" + resp.json()["response"]
print(raw_output)

Same question without the chat template, and the model rambles.

**Goal:** see why the template matters. Without the `<|im_start|>...<|im_end|>` structure, the model treats the prompt as a plain text completion. It isn't in "assistant turn" mode, so there's no end-of-turn signal it's trained to emit. It just keeps writing until we cut it off.

In [ ]:
# CELL: no-template
# Without the chat template the model has no assistant turn to end, so it never
# emits a stop token: it just keeps going. Streaming returns one JSON object per
# token; we collect for 3 seconds, then stop reading (the server handles the
# dropped connection fine).
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "What is 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

print("".join(tokens))

`ollama.chat()` renders the template and parses the output for you.

**Goal:** see how the convenient API does all three layers automatically: your `messages` get wrapped in `<|im_start|>...<|im_end|>` tokens, the LLM produces text, and the parser splits out `thinking` and `content`.

In [ ]:
# CELL: ollama.chat
# ollama.chat() does the same thing, but:
# 1. Builds the prompt with special tokens automatically (RENDERER)
# 2. Parses <think>...</think> from the output into the "thinking" field (PARSER)
# 3. The rest becomes "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why. Each demo cell above starts with a `# CELL: <name>` label so you can find it.

1. Copy the **`raw-API prefill`** cell. Remove the `<think>\n` from the end of the `prompt`, and also drop the `raw_output = "<think>\n" + ...` reconstruction (just print `resp.json()["response"]`). *You should see:* the `<think>...</think>` block gone from the output, because you no longer pre-fill it.
2. Copy the **`no-template`** cell. Change its prompt to `"Answer once, then stop: What is 2 + 2?"`. *You should see:* it likely still runs away. What does that tell you about what actually stops generation? (In raw mode nothing emits the `<|im_end|>` stop token, no matter what the text asks.)


In [ ]:
# Your Turn: Part 1
# Q1: copy the "raw-API prefill" cell above.  Q2: copy the "no-template" cell above.



In [ ]:
# === Answers: Part 1 (keep each tag, write your answer after the colon) ===
# P1.Q1: 
# P1.Q2: 


## Part 2: Statelessness

The chat API is stateless. The model only sees the `messages` we send in each call.

**Context window:** every call ships the full `messages` list (system prompt + all prior turns + tool results) through the model's fixed token budget, called the **context window** and set by `num_ctx`. Typical models support 4k to 128k tokens (a token ≈ ¾ of an English word). The model re-reads everything from scratch each call. As the chat grows, history fills the window, and eventually old messages must be dropped or summarized. This is why agents need memory strategies.

```
  Turn 1:  [sys][u1]                                  ─── num_ctx limit
  Turn 2:  [sys][u1][a1][u2]                                        │
  Turn 5:  [sys][u1][a1][u2][a2][u3][a3][u4][a4][u5]                │
  Turn 12: ████████████████████████████████████████████  ← FULL ────┘
           └─────────── must drop or summarize ───────────┘
```

<details><summary>Wait, does the model really re-read everything every call?</summary>

Conceptually yes, and that is why statelessness shapes how you write the loop. Under the hood Ollama reuses computation when consecutive calls share a prefix. This optimization is called a **KV cache** (short for Key/Value cache): inside the transformer, every token produces two tensors named "key" and "value" that the attention layer needs to consult. Ollama stores those tensors for tokens already sent, and reuses them on the next call instead of recomputing them.

The model itself still has no awareness across calls; the server just skips work it already did. From your code's perspective nothing changes, you still send the full `messages` list every time.

</details>

Two stateless calls. The model has no memory between them. **Goal:** confirm the API has no session state; what the model knows comes from `messages` alone.

In [ ]:
# CELL: two-calls
# Without conversation history: two separate calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "What is my name?"}])
print("Call 2:", r2.message.content)

Same calls, but with the previous turn passed back as history. **Goal:** see that "memory" is just history we assemble and resend.

In [ ]:
# CELL: with-history
# With conversation history: same two calls, but we pass the history along
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "My name is Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "What is my name?"},
])
print("Call 2:", r2.message.content)

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why. All three tasks start from the **`with-history`** cell above.

1. Remove the `{"role": "assistant", ...}` entry from the second call's message list. *You should see:* the model no longer reliably knows the name, because you deleted the turn that carried it.
2. Replace `r1.message.content` in the history with the literal string `"Nice to meet you, Anna!"`. *You should see:* the model now thinks your name is Anna: it only "knows" what the history says, not what really happened.
3. Move the fact `"My name is Max."` out of the first `user` message and into a `system` message at the start of the list (`{"role": "system", "content": "..."}`), and send only `"What is my name?"` as the user turn (no assistant history). *You should see:* it still answers Max, because the fact is now standing context. Which role do instructions and stable facts conventionally live in, and why is that different from a fact the user states mid-conversation?


In [ ]:
# Your Turn: Part 2
# All three tasks: copy the "with-history" cell above.



In [ ]:
# === Answers: Part 2 (keep each tag, write your answer after the colon) ===
# P2.Q1: 
# P2.Q2: 
# P2.Q3: 


## Part 3: Tool Calling

An LLM always only generates text. Tool calling is not a special feature, but a **convention for structured output**: the model was trained to output JSON instead of free text for certain prompts.

Tool calling by hand: a system prompt asks the model for JSON. **Goal:** see that tool calling is not a special model feature, just structured text output.

In [ ]:
# CELL: tool-by-hand
# "Tool calling" without the tools= parameter, using only a prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'When you need to calculate, respond ONLY with this JSON:\n{"tool": "calculator", "expression": "<expression>"}\nNo other text.'},
        {"role": "user", "content": "Calculate 17 * 23"},
    ],
)

print("=== Raw Output (just text) ===")
print(response.message.content)

Parse that JSON ourselves and run the expression. **Goal:** see that the LLM only decides what to call; execution is on us.

In [ ]:
# CELL: parse-and-run
# We can parse the text output ourselves and execute the tool
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# This is tool calling: the model outputs structured text, we parse and execute.
print(parsed["expression"])
result = eval(parsed["expression"])
print("Result:", result)

This works, but it's fragile. Both this approach and the `tools=` parameter end up putting tool info into the prompt, so why is hand-rolling fragile?

- **By hand:** you invent the format. The model treats it as a generic instruction and often deviates: it adds preamble ("Here's the JSON: ..."), wraps the output in code fences, mangles keys, or refuses on edge cases. You then have to robustly parse whatever it emits.
- **With `tools=`:** ollama injects the tool info in the **exact format the model was fine-tuned on**. Qwen3.5 has seen thousands of (tool_def → tool_call) examples in this format and learned to emit a specific, reliably-parseable structure. Ollama's parser then pulls the call out into a structured `tool_calls` field.

From here on we use `tools=`. The **tool definition** is the JSON schema describing the function. The **tool implementation** is the Python function we execute. They're connected via `tool_map`.

Tool definition: the JSON schema we hand to the model. **Goal:** see what the model needs to know about an available function (name, description, parameter shape).

In [ ]:
# CELL: tool-definition
# Tool definition: describes to the model which function is available
# Reference: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

With `tools=`, the model returns `tool_calls` instead of text. **Goal:** see how the structured output now appears as a parsed field, not as raw JSON in `content`.

In [ ]:
# CELL: tools= call
# Call the LLM with the tool definition, inspect the full response
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Calculate 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

In the response we see:
- `content` is empty: the model did not return any text
- `tool_calls` is populated: the model wants to call `calculator` with `{"expression": "17 * 23"}`

The model did not calculate anything. It returned **which function** it wants to call with **which arguments**. Now we need the Python function that actually executes it.

Tool implementation: the Python function we actually run. **Goal:** distinguish the schema (what the model sees) from the code (what we execute).

In [ ]:
# CELL: tool-implementation
# Tool implementation: the Python function we execute
import numexpr, math
# numexpr evaluates math safely (no arbitrary code execution, unlike eval()).

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: call directly
print(calculator("17 * 23"))

Dispatch: look up the function by name, call it with the model's arguments. **Goal:** see how a simple name→function dict wires the model's choice to real code.

In [ ]:
# CELL: dispatch
# Connection: how do we find the right function for a tool call?
# The name in the tool call ("calculator") must match the key in tool_map.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool call from the response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # Look up the calculator function
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why. Where a task touches two cells, run them **in order** so your edit takes effect.

1. Copy **both** the **`tool-definition`** and **`tools= call`** cells. Set `description` to an empty string `""`, then run them in order (definition first). *You should see:* it often still calls the tool, because the name and parameter schema also signal usage; the description matters most when tools compete or the task is ambiguous.
2. Copy the **`tools= call`** cell. Change the user prompt to `"What is 2 + 2?"`. *You should see:* compare `content` vs `tool_calls`. Does the model reach for the tool on trivial math, or just answer directly?
3. Copy the **`tool-definition`**, **`tools= call`**, and **`dispatch`** cells. Rename the tool in `calculator_tool` from `"calculator"` to `"math_thing"`, but leave `tool_map = {"calculator": calculator}` unchanged. Run all three in order (definition, then a fresh `tools=` call, then the dispatch). *You should see:* a `KeyError: 'math_thing'` at the `tool_map[tc.function.name]` line in the dispatch, because the model's tool name no longer matches a key in `tool_map`.


In [ ]:
# Your Turn: Part 3
# Q1: "tool-definition" + "tools= call" cells.  Q2: "tools= call" cell.
# Q3: "tool-definition" + "tools= call" + "dispatch" cells. Run multi-cell tasks in order.



In [ ]:
# === Answers: Part 3 (keep each tag, write your answer after the colon) ===
# P3.Q1: 
# P3.Q2: 
# P3.Q3: 


## Part 4: Tool-Use Loop

We have a tool result, but the API is stateless: the model knows nothing about the execution. We must include the result as a `{"role": "tool"}` message in the history and call the model again.

Full loop: model requests tool → we execute → we send the result back → model answers. **Goal:** see the complete cycle an agent automates.

```
   ┌─────────────┐
   │ ollama.chat │ ←───────────────┐
   └──────┬──────┘                 │
          ↓                        │
    tool_calls? ─── No ──→  done   │
          │                        │
          │ Yes                    │
          ↓                        │
   ┌──────────────┐                │
   │ run tool,    │ ───────────────┘
   │ append result│
   └──────────────┘
```

In [ ]:
# CELL: tool-use loop
# Complete tool-use loop
messages = [{"role": "user", "content": "Calculate 17 * 23"}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back. First append the assistant's own tool-call
    # message (model_dump() turns the response object into a plain dict, keeping
    # its tool_calls) so the tool result below has a request to answer; then the
    # result itself.
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

Sanity check: when the model declines, `tool_calls` is `None`. **Goal:** confirm the model decides if and when to use a tool; we just check the field and skip the dispatch.

In [ ]:
# CELL: answer-directly
# Sanity check: what happens when the model decides AGAINST using a tool?
messages = [{"role": "user", "content": "Calculate 17 * 23. Answer directly, do NOT use a tool."}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why.

1. Copy the **`tool-use loop`** cell. Remove the line `messages.append(response.message.model_dump())` (keep the `{"role": "tool", ...}` append). *You should see:* the final answer gets confused or errors, because you sent a tool *result* with no preceding tool *request* for it to answer.
2. Copy the **`tool-use loop`** cell. Change the prompt to `"Calculate 17 * 23, then divide the result by 2."` and turn the single `if` into a capped loop so the model can call the tool more than once:
   ```python
   messages = [{"role": "user", "content": "Calculate 17 * 23, then divide the result by 2."}]
   for _ in range(5):  # safety cap
       response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
       if not response.message.tool_calls:
           break
       tc = response.message.tool_calls[0]
       result = tool_map[tc.function.name](**tc.function.arguments)
       messages.append(response.message.model_dump())
       messages.append({"role": "tool", "content": result})
   print(response.message.content)
   ```
   *You should see:* two sequential tool calls (first the multiply, then the divide), not one combined expression.
3. Copy the **`answer-directly`** cell. Change its prompt to `"Tell me a short joke."` (leave `tools=calculator_tool` unchanged). *You should see:* the model ignores the calculator: `tool_calls` is `None` and the joke is in `content`. What does that tell you about how tool selection is decided?


In [ ]:
# Your Turn: Part 4
# Q1 & Q2: copy the "tool-use loop" cell above.  Q3: copy the "answer-directly" cell above.



In [ ]:
# === Answers: Part 4 (keep each tag, write your answer after the colon) ===
# P4.Q1: 
# P4.Q2: 
# P4.Q3: 


## Summary

1. `ollama.chat()` takes a `messages` list and returns a `ChatResponse` object.
2. The API is stateless. The model only sees the messages we send in each call.
3. With `tools=`, the model can return tool calls. We check for this with `if response.message.tool_calls`.
4. The tool definition (JSON object with name, description, parameter schema) tells the model what is available. The tool implementation (Python function) is executed by us. They are connected via `tool_map`.
5. The tool-use loop (call LLM → check → execute tool → send result back → call LLM again) is what an agent automates.

→ **Exercise 2**: implement this loop.